# 随机厄米矩阵特征值分析演示
# Random Hermitian Matrix Eigenvalue Analysis Demo

这个notebook演示了随机矩阵理论中的经典问题，包括：
- **LU分解的标度行为分析**
- **特征值对角化**
- **归一化间距计算**
- **Wigner-Dyson统计验证**
- **局部平均方法比较**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hermitian_matrix_analysis import (
    HermitianMatrixAnalyzer,
    analyze_lu_scaling,
    plot_spacing_distribution,
    plot_lu_scaling,
    compare_local_averaging_methods
)

# 设置matplotlib支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表样式
plt.style.use('seaborn-v0_8-darkgrid')

print("导入成功！")

## 任务 a: LU分解和标度分析

LU分解是将矩阵A分解为下三角矩阵L和上三角矩阵U的乘积：PA = LU

理论上，LU分解的时间复杂度为 **O(N³)**。我们通过测试不同大小的矩阵来验证这个标度行为。

In [ ]:
# 测试不同大小的矩阵
print("执行 LU 分解标度分析...")
print("="*60)

# 从小到大测试
N_values = [50, 100, 200, 400, 800]
scaling_results = analyze_lu_scaling(N_values, num_trials=3, seed=42)

In [ ]:
# 可视化标度行为
fig = plot_lu_scaling(scaling_results)
plt.show()

print("\n结论: LU分解的时间复杂度确实为 O(N³)")

## 任务 b & c: 对角化和归一化间距

### 理论背景

对于随机厄米矩阵，特征值的统计性质遵循 **Wigner-Dyson 统计**，表现出显著的**能级排斥**现象。

归一化间距定义为：
$$s_i = \frac{\Delta\lambda_i}{\bar{\Delta\lambda}}$$

其中：
- $\Delta\lambda_i = \lambda_{i+1} - \lambda_i$ 是相邻特征值间距
- $\bar{\Delta\lambda}$ 是平均间距

对于高斯酉系综(GUE)，理论预测：
$$P(s) = \frac{32}{\pi^2} s^2 e^{-\frac{4s^2}{\pi}}$$

In [ ]:
# 创建分析器并生成随机厄米矩阵
N = 1000  # 矩阵大小
print(f"生成 {N}×{N} 随机厄米矩阵...")

analyzer = HermitianMatrixAnalyzer(N, seed=42)
A = analyzer.generate_hermitian_matrix()

print(f"矩阵形状: {A.shape}")
print(f"矩阵是厄米的: {np.allclose(A, A.conj().T)}")
print(f"矩阵迹: {np.trace(A):.4f}")

In [ ]:
# 对角化矩阵
print("对角化矩阵...")
eigenvalues, eigenvectors = analyzer.diagonalize_matrix()

print(f"特征值个数: {len(eigenvalues)}")
print(f"特征值范围: [{eigenvalues[0]:.4f}, {eigenvalues[-1]:.4f}]")
print(f"特征值平均值: {np.mean(eigenvalues):.4f}")

# 验证对角化
# A·v = λ·v
test_idx = N // 2
Av = A @ eigenvectors[:, test_idx]
lambda_v = eigenvalues[test_idx] * eigenvectors[:, test_idx]
print(f"\n对角化验证 (索引 {test_idx}):")
print(f"  ||A·v - λ·v|| = {np.linalg.norm(Av - lambda_v):.2e}")

In [ ]:
# 绘制特征值分布
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(eigenvalues, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('特征值 λ')
plt.ylabel('频数')
plt.title('特征值直方图')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(eigenvalues, 'o-', markersize=2, alpha=0.6)
plt.xlabel('索引 i')
plt.ylabel('特征值 λᵢ')
plt.title('排序后的特征值')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 计算归一化间距（使用全局平均）
print("计算归一化间距...")
results = analyzer.calculate_normalized_spacings()

normalized_spacings = results['global']

print(f"\n归一化间距统计:")
print(f"  间距个数: {len(normalized_spacings)}")
print(f"  平均值: {np.mean(normalized_spacings):.4f} (理论值 ≈ 1.0)")
print(f"  标准差: {np.std(normalized_spacings):.4f}")
print(f"  中位数: {np.median(normalized_spacings):.4f}")
print(f"  最小值: {np.min(normalized_spacings):.4f}")
print(f"  最大值: {np.max(normalized_spacings):.4f}")

In [ ]:
# 绘制归一化间距分布并与理论对比
fig = plot_spacing_distribution(
    normalized_spacings,
    ensemble='GUE',
    bins=60,
    title=f'归一化特征值间距分布 (N={N})'
)
plt.show()

print("\n观察:")
print("1. 在 s ≈ 0 附近，概率密度接近零 → 能级排斥")
print("2. 分布与 Wigner-Dyson (GUE) 理论吻合良好")
print("3. 显著偏离 Poisson 分布 → 特征值之间存在关联")

## 任务 d: 局部平均分析

不同的归一化方法可能影响间距统计。我们比较以下方法：

1. **全局平均**: 使用所有间距的平均值
2. **局部平均**: 在每个特征值周围使用局部窗口计算平均值

窗口大小的选择：
- N/100: 非常局部（1% 的能级）
- N/50: 局部（2% 的能级）
- N/10: 中等（10% 的能级）
- N/5: 较大（20% 的能级）
- N: 全局

In [ ]:
# 定义窗口大小
window_sizes = [
    max(2, N // 100),  # 1%
    max(2, N // 50),   # 2%
    max(2, N // 10),   # 10%
    max(2, N // 5),    # 20%
    N                  # 100%
]

print("窗口大小:")
for ws in window_sizes:
    print(f"  {ws:4d} ({100*ws/N:5.1f}% of N)")

In [ ]:
# 比较不同局部平均方法
fig = compare_local_averaging_methods(analyzer, window_sizes)
plt.show()

print("\n观察:")
print("1. 小窗口(局部): 可以捕捉特征值密度的局部变化")
print("2. 大窗口(全局): 提供更平滑的统计，但可能掩盖局部特征")
print("3. 所有方法都显示出 Wigner-Dyson 统计的基本特征")
print("4. 对于均匀分布的特征值，不同方法的差异较小")

In [ ]:
# 定量比较不同方法
print("\n定量比较不同归一化方法:")
print("="*60)

all_results = analyzer.calculate_normalized_spacings(window_sizes=window_sizes)

for key, spacings in all_results.items():
    mean = np.mean(spacings)
    std = np.std(spacings)
    median = np.median(spacings)
    
    # 计算与理论分布的 KS 距离（粗略估计）
    sorted_spacings = np.sort(spacings)
    n_points = len(sorted_spacings)
    
    print(f"\n{key}:")
    print(f"  均值: {mean:.4f}")
    print(f"  标准差: {std:.4f}")
    print(f"  中位数: {median:.4f}")

## 额外分析: 不同矩阵大小的影响

让我们测试不同的矩阵大小，看看 Wigner-Dyson 统计是否对 N 具有鲁棒性。

In [ ]:
# 测试不同的 N 值
N_test_values = [100, 500, 1000, 2000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# 理论曲线
s_theory = np.linspace(0, 5, 200)

for idx, N_test in enumerate(N_test_values):
    print(f"处理 N = {N_test}...")
    
    # 创建分析器
    test_analyzer = HermitianMatrixAnalyzer(N_test, seed=42)
    test_analyzer.generate_hermitian_matrix()
    test_analyzer.diagonalize_matrix()
    
    # 计算归一化间距
    test_results = test_analyzer.calculate_normalized_spacings()
    test_spacings = test_results['global']
    
    # 绘图
    ax = axes[idx]
    ax.hist(test_spacings, bins=50, density=True, alpha=0.6,
           edgecolor='black', label='数值结果')
    
    wd_dist = test_analyzer.wigner_dyson_distribution(s_theory, 'GUE')
    ax.plot(s_theory, wd_dist, 'r-', linewidth=2, label='Wigner-Dyson (GUE)')
    
    ax.set_xlabel('归一化间距 s')
    ax.set_ylabel('概率密度 P(s)')
    ax.set_title(f'N = {N_test}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n结论: 随着 N 增大，数值结果越来越接近理论预测")

## 物理意义总结

### 能级排斥（Level Repulsion）

在随机厄米矩阵中，特征值（能级）表现出**相互排斥**的现象：
- 两个能级非常接近的概率极低 (P(s→0) → 0)
- 这与**量子混沌系统**的能级统计一致
- 与可积系统的 Poisson 统计形成对比

### 应用领域

1. **核物理**: 原子核能级的统计性质
2. **量子混沌**: 判断系统的混沌性质
3. **凝聚态物理**: 无序系统的能谱
4. **数论**: 黎曼 zeta 函数的零点统计
5. **通信理论**: MIMO 系统的信道容量

### 标度行为

- **LU 分解**: O(N³) 复杂度
- **特征值分解**: O(N³) 复杂度
- **Wigner 半圆律**: 特征值密度 ρ(λ) ∝ √(4-λ²/σ²)
- **普适性**: 结果对矩阵系综的细节不敏感

## 参考文献

1. Mehta, M. L. (2004). *Random Matrices*. Academic Press.
2. Haake, F. (2010). *Quantum Signatures of Chaos*. Springer.
3. Forrester, P. J. (2010). *Log-Gases and Random Matrices*. Princeton University Press.
4. Tao, T. (2012). *Topics in Random Matrix Theory*. American Mathematical Society.